In [ ]:
!pip install pyspark

**Part 1: Clustering**

In [ ]:
import time
import random
from pyspark import SparkContext
from pyspark.mllib.linalg import Vectors

# Initialize SparkContext to use PySpark locally
try:
    sc = SparkContext.getOrCreate()
except Exception as e:
    print(f"Spark context could not be created: {e}")

# readVectorsSeq Function
def readVectorsSeq(filename):
    """
    Transforms a text file of feature points into a list of Vectors.
    """
    vectors_list = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                # Features are separated by commas
                features = [float(x) for x in line.split(',')]
                # Transform array of doubles into a Vector instance
                vectors_list.append(Vectors.dense(features))
    return vectors_list

def _min_sqdist(point, centers):
    """
    Helper function to calculate the squared L2-distance between a point
    and its closest center.
    """
    min_dist = float('inf')
    for c in centers:
        # Note: PySpark's Python API uses point.squared_distance(c)
        # instead of the Java/Scala Vectors.sqdist(point, c)
        dist = point.squared_distance(c)
        if dist < min_dist:
            min_dist = dist
    return min_dist

# kcenter Function (Farthest-First Traversal)
def kcenter(P, k):
    """
    Receives a set of points P and an integer k, and returns a set C of k centers
    computed by the Farthest-First Traversal algorithm.
    Runs in time O(|P| x k).
    """
    if not P or k <= 0:
        return []

    # Arbitrarily pick the first point as the first center
    centers = [P[0]]

    for _ in range(1, k):
        max_dist = -1.0
        farthest_point = None

        # Find the point that is farthest away from its closest center
        for point in P:
            dist = _min_sqdist(point, centers)
            if dist > max_dist:
                max_dist = dist
                farthest_point = point

        centers.append(farthest_point)

    return centers

# kmeansPP Function
def kmeansPP(P, k):
    """
    Receives a set of points P and an integer k, and returns a set C of k centers
    computed with the kmeans++ algorithm.
    Runs in time O(|P| x k).
    """
    if not P or k <= 0:
        return []

    # Choose one center uniformly at random
    centers = [random.choice(P)]

    for _ in range(1, k):
        distances = []
        total_dist = 0.0

        for point in P:
            dist = _min_sqdist(point, centers)
            distances.append(dist)
            total_dist += dist

        # Pick the next center with probability proportional to its squared distance
        r = random.uniform(0, total_dist)
        cumulative_dist = 0.0
        for i, dist in enumerate(distances):
            cumulative_dist += dist
            if cumulative_dist >= r:
                centers.append(P[i])
                break

    return centers

# 4. kmeansObj Function
def kmeansObj(P, C):
    """
    Returns the average squared distance of a point of P from its closest center.
    """
    if not P or not C:
        return 0.0

    total_sqdist = 0.0
    for point in P:
        total_sqdist += _min_sqdist(point, C)

    # Divide by the number of points of P
    return total_sqdist / len(P)


# Main Python Program execution block
def run_clustering_experiments(filename, k, k1):
    """
    Main program to evaluate kcenter and kmeansPP algorithms.
    """
    if k >= k1:
        print("Requirement error: k must be less than k1")
        return

    print("Loading dataset...")
    P = readVectorsSeq(filename)

    if not P:
        print("Error reading dataset or dataset is empty.")
        return

    print(f"Loaded {len(P)} points.")

    # Step 1: Run kcenter(P, k) printing its running time
    print(f"\n--- 1. kcenter (k={k}) ---")
    start_time = time.time()
    C_kcenter = kcenter(P, k)
    end_time = time.time()
    print(f"kcenter execution time: {end_time - start_time:.4f} seconds")

    # Step 2: Run kmeansPP(P, k) and evaluate objective function
    print(f"\n--- 2. kmeansPP (k={k}) ---")
    C_kmeansPP = kmeansPP(P, k)
    obj_kmeansPP = kmeansObj(P, C_kmeansPP)
    print(f"Returned value (kmeansObj for kmeansPP): {obj_kmeansPP:.4f}")

    # Step 3: Run kcenter(P, k1) to obtain X, then kmeansPP(X, k)
    print(f"\n--- 3. Coreset Evaluation (k1={k1}, k={k}) ---")

    # Obtain set of k1 centers X
    X = kcenter(P, k1)

    # Extract set of k centers C from X
    C_coreset = kmeansPP(X, k)

    # Calculate kmeansObj for P with centers C and print
    obj_coreset = kmeansObj(P, C_coreset)
    print(f"Returned value (kmeansObj with coreset X): {obj_coreset:.4f}")

    print("\nExperiment Complete.")

# Execute the experiment
dataset_filename = 'spambase.data'
run_clustering_experiments(dataset_filename, k=10, k1=50)

Loading dataset...
Loaded 4601 points.

--- 1. kcenter (k=10) ---
kcenter execution time: 0.6869 seconds

--- 2. kmeansPP (k=10) ---
Returned value (kmeansObj for kmeansPP): 28156.6201

--- 3. Coreset Evaluation (k1=50, k=10) ---
Returned value (kmeansObj with coreset X): 341206.3424

Experiment Complete.


**Part 2: Web-Search**


In [ ]:
import os

# --- Helper Configurations based on instructions ---
CONNECTORS = {"a", "an", "the", "they", "these", "this", "for", "is", "are",
              "was", "of", "or", "and", "does", "will", "whose"}

PLURALS = {"stacks": "stack", "structures": "structure", "applications": "application"}

def preprocess_text(text):
    """Replaces punctuation with spaces, converts to lowercase, and splits into words."""
    punctuation = "{}[]<>=().,;\'\"?#!:"
    for p in punctuation:
        text = text.replace(p, ' ')
    return text.lower().split()

def normalize_word(word):
    """Converts plural forms to singular forms based on the predefined dictionary."""
    return PLURALS.get(word, word)

# --- Class Implementations ---

class MySet:
    def __init__(self):
        self.elements = set()

    def addElement(self, element):
        self.elements.add(element)

    def union(self, otherSet):
        result = MySet()
        result.elements = self.elements.union(otherSet.elements)
        return result

    def intersection(self, otherSet):
        result = MySet()
        result.elements = self.elements.intersection(otherSet.elements)
        return result

    def get_elements(self):
        return self.elements

class Position:
    def __init__(self, p, wordIndex):
        self.p = p # PageEntry object
        self.wordIndex = wordIndex

    def getPageEntry(self):
        return self.p

    def getWordIndex(self):
        return self.wordIndex

class WordEntry:
    def __init__(self, word):
        self.word = word
        self.positions = []

    def addPosition(self, position):
        self.positions.append(position)

    def addPositions(self, positions):
        self.positions.extend(positions)

    def getAllPositionsForThisWord(self):
        return self.positions

    def getTermFrequency(self, word):
        if not self.positions:
            return 0.0
        # Gets the total words from the page associated with the first position
        page = self.positions[0].getPageEntry()
        return len(self.positions) / page.total_words

class PageIndex:
    def __init__(self):
        self.word_entries = {}

    def addPositionForWord(self, str_word, p):
        if str_word not in self.word_entries:
            self.word_entries[str_word] = WordEntry(str_word)
        self.word_entries[str_word].addPosition(p)

    def getWordEntries(self):
        return list(self.word_entries.values())

class PageEntry:
    def __init__(self, pageName):
        self.pageName = pageName
        self.pageIndex = PageIndex()
        self.total_words = 0

        # Ensure the webpages directory is targeted
        filepath = os.path.join("webpages", pageName)

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
        except FileNotFoundError:
            print(f"Error: File {filepath} not found.")
            return

        words = preprocess_text(content)
        self.total_words = len(words)

        # Process words with 1-based indexing for position
        for i, w in enumerate(words):
            word_index = i + 1
            if w not in CONNECTORS:
                norm_w = normalize_word(w)
                pos = Position(self, word_index)
                self.pageIndex.addPositionForWord(norm_w, pos)

    def getPageIndex(self):
        return self.pageIndex

class MyHashTable:
    def __init__(self):
        # We use a python dict to act as the core hash table structure
        self.table = {}

    def getHashIndex(self, str_word):
        # In Python, strings are hashable and act as direct dictionary keys
        return str_word

    def addPositionsForWord(self, w):
        idx = self.getHashIndex(w.word)
        if idx not in self.table:
            self.table[idx] = WordEntry(w.word)
        self.table[idx].addPositions(w.getAllPositionsForThisWord())

class InvertedPageIndex:
    def __init__(self):
        self.hash_table = MyHashTable()
        self.pages = {} # Keep track of loaded pages

    def addPage(self, p):
        self.pages[p.pageName] = p
        for we in p.getPageIndex().getWordEntries():
            self.hash_table.addPositionsForWord(we)

    def getPagesWhichContainWord(self, str_word):
        norm_word = normalize_word(str_word.lower())
        idx = self.hash_table.getHashIndex(norm_word)
        pages_set = MySet()

        if idx in self.hash_table.table:
            for pos in self.hash_table.table[idx].getAllPositionsForThisWord():
                pages_set.addElement(pos.getPageEntry())

        return pages_set

class SearchEngine:
    def __init__(self):
        self.inv_index = InvertedPageIndex()

    def performAction(self, actionMessage):
        parts = actionMessage.strip().split()
        if not parts:
            return

        action = parts[0]

        if action == "addPage":
            page_name = parts[1]
            p = PageEntry(page_name)
            self.inv_index.addPage(p)

        elif action == "queryFindPagesWhichContainWord":
            word = parts[1]
            pages_set = self.inv_index.getPagesWhichContainWord(word)
            elements = pages_set.get_elements()

            if not elements:
                print(f"No webpage contains word {word}")
            else:
                names = sorted([p.pageName for p in elements])
                print(", ".join(names))

        elif action == "queryFindPositionsOfWordInAPage":
            word = parts[1]
            page_name = parts[2]
            norm_word = normalize_word(word.lower())

            if page_name not in self.inv_index.pages:
                print(f"No webpage {page_name} found")
                return

            target_page = self.inv_index.pages[page_name]
            page_index = target_page.getPageIndex()

            if norm_word in page_index.word_entries:
                positions = page_index.word_entries[norm_word].getAllPositionsForThisWord()
                indices = [str(pos.getWordIndex()) for pos in positions]
                print(", ".join(indices))
            else:
                print(f"Webpage {page_name} does not contain word {word}")


# --- Execution Block ---
if __name__ == "__main__":
    engine = SearchEngine()

    actions_file = "actions.txt"
    print("--- Running Actions from actions.txt ---")

    try:
        with open(actions_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    # Optional: Print the action being performed for better visibility
                    # print(f"> {line.strip()}")
                    engine.performAction(line)
    except FileNotFoundError:
        print(f"Make sure '{actions_file}' is uploaded to the root directory.")

--- Running Actions from actions.txt ---
No webpage contains word delhi
stack_datastructure_wiki
stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines
No webpage contains word allain
stack_cprogramming
stack_cprogramming
stack_cprogramming
stack_oracle
stack_cprogramming, stack_datastructure_wiki, stackoverflow
stackmagazine


**Part 3: PageRank on Spark**

In [3]:
from pyspark import SparkContext

# Initialize Spark Context
try:
    sc = SparkContext.getOrCreate()
except Exception as e:
    print(f"Spark context could not be created: {e}")

def compute_contributions(urls, rank):
    """Calculates URL contributions to the rank of other URLs."""
    num_urls = len(urls)
    for url in urls:
        yield (url, rank / num_urls)

def run_pagerank(filename, iterations=40, beta=0.8):
    print(f"\n--- Running PageRank on {filename} ---")

    # Load the dataset
    try:
        lines = sc.textFile(filename)
    except Exception as e:
        print(f"Error loading {filename}. Is it uploaded to Colab?")
        return

    # Parse the graph
    edges = lines.map(lambda line: tuple(line.strip().split())) \
                 .filter(lambda x: len(x) == 2) \
                 .distinct()

    # Find all unique nodes and total count 'n'
    nodes = edges.map(lambda x: x[0]).union(edges.map(lambda x: x[1])).distinct()
    n = nodes.count()
    print(f"Total nodes (n): {n}")

    # Create Adjacency List: (source, [dest1, dest2, ...])
    links = edges.groupByKey().mapValues(list).cache()

    # Initialize ranks to 1/n
    ranks = nodes.map(lambda node: (node, 1.0 / n))
    base_nodes = nodes.map(lambda node: (node, 0.0)).cache()

    # Iterative PageRank Algorithm
    print(f"Running {iterations} iterations with beta={beta}...")
    for iteration in range(iterations):
        contribs = links.join(ranks).flatMap(
            lambda x: compute_contributions(x[1][0], x[1][1])
        )

        sum_contribs = contribs.reduceByKey(lambda x, y: x + y)

        # Apply the PageRank formula: r = (1 - beta)/n + beta * M * r [cite: 186]
        ranks = base_nodes.leftOuterJoin(sum_contribs).mapValues(
            lambda x: (1.0 - beta) / n + beta * (x[1] if x[1] is not None else 0.0)
        )

        if (iteration + 1) % 5 == 0:
            local_ranks = ranks.collect()
            ranks = sc.parallelize(local_ranks)
            print(f"Completed iteration {iteration + 1}...")

    # Collect and sort results
    top_5 = ranks.sortBy(lambda x: x[1], ascending=False).take(5)
    bottom_5 = ranks.sortBy(lambda x: x[1], ascending=True).take(5)

    print("\nTop 5 Nodes (Highest Scores):")
    for node, rank in top_5:
        print(f"Node: {node}, Score: {rank:.6f}")

    print("\nBottom 5 Nodes (Lowest Scores):")
    for node, rank in bottom_5:
        print(f"Node: {node}, Score: {rank:.6f}")

# --- Execution ---
run_pagerank("small.txt", iterations=40, beta=0.8)


--- Running PageRank on small.txt ---
Total nodes (n): 100
Running 40 iterations with beta=0.8...
Completed iteration 5...
Completed iteration 10...
Completed iteration 15...
Completed iteration 20...
Completed iteration 25...
Completed iteration 30...
Completed iteration 35...
Completed iteration 40...

Top 5 Nodes (Highest Scores):
Node: 53, Score: 0.035731
Node: 14, Score: 0.034171
Node: 40, Score: 0.033630
Node: 1, Score: 0.030006
Node: 27, Score: 0.029720

Bottom 5 Nodes (Lowest Scores):
Node: 85, Score: 0.003410
Node: 59, Score: 0.003670
Node: 81, Score: 0.003695
Node: 37, Score: 0.003808
Node: 89, Score: 0.003922


In [4]:
run_pagerank("whole.txt", iterations=40, beta=0.8)


--- Running PageRank on whole.txt ---
Total nodes (n): 1000
Running 40 iterations with beta=0.8...
Completed iteration 5...
Completed iteration 10...
Completed iteration 15...
Completed iteration 20...
Completed iteration 25...
Completed iteration 30...
Completed iteration 35...
Completed iteration 40...

Top 5 Nodes (Highest Scores):
Node: 263, Score: 0.002020
Node: 537, Score: 0.001943
Node: 965, Score: 0.001925
Node: 243, Score: 0.001853
Node: 285, Score: 0.001827

Bottom 5 Nodes (Lowest Scores):
Node: 558, Score: 0.000329
Node: 93, Score: 0.000351
Node: 62, Score: 0.000353
Node: 424, Score: 0.000355
Node: 408, Score: 0.000388
